In [1]:
import numpy as np
import pandas as pd
import os

# -------------------
# CONFIG
# -------------------
BASE_PATH = "../../../data"
DATA_PATHS = {
    "training": {
        "engine_start":   f"{BASE_PATH}/training/EngineStart_training_X.npy",
        "engine_off":     f"{BASE_PATH}/training/EngineOff_training_X.npy",
        "normal":         f"{BASE_PATH}/training/NormalLoad_training_X.npy",
        "high":           f"{BASE_PATH}/training/HighLoad_training_X.npy",
        "critical":       f"{BASE_PATH}/training/CriticalLoad_training_X.npy",
    },
    "test": {
        "engine_start":   f"{BASE_PATH}/test/EngineStart_test_X.npy",
        "engine_off":     f"{BASE_PATH}/test/EngineOff_test_X.npy",
        "normal":         f"{BASE_PATH}/test/NormalLoad_test_X.npy",
        "high":           f"{BASE_PATH}/test/HighLoad_test_X.npy",
        "critical":       f"{BASE_PATH}/test/CriticalLoad_test_X.npy",
    },
    "final test": {
        "engine_start":   f"{BASE_PATH}/final test/EngineStart_final test_X.npy",
        "engine_off":     f"{BASE_PATH}/final test/EngineOff_final test_X.npy",
        "normal":         f"{BASE_PATH}/final test/NormalLoad_final test_X.npy",
        "high":           f"{BASE_PATH}/final test/HighLoad_final test_X.npy",
        "critical":       f"{BASE_PATH}/final test/CriticalLoad_final test_X.npy",
    }
}

FEATURES = ["Temperature", "Pressure", "RPM", "Vibration"]
SEED = 42
np.random.seed(SEED)

# --- probabilities for mixing ---
P_MIX_START_IN_LOAD = 0.30   # 30% chance load windows contain Engine Start rows
P_MIX_LOAD_IN_OFF   = 0.30   # 30% chance off windows contain Load rows

def build_router_dataset(split_name, paths):
    print(f"\n--- Building router dataset for split: {split_name} ---")

    # Load base arrays
    start_data    = np.load(paths["engine_start"])     # (Ns,4,4)
    off_data      = np.load(paths["engine_off"])       # (No,1,4) or (No,4)
    normal_data   = np.load(paths["normal"])           # (Nn,1,4) or (Nn,4)
    high_data     = np.load(paths["high"])
    critical_data = np.load(paths["critical"])

    # Helper: flatten to (rows, 4)
    def to_rows(arr):
        return arr.reshape(-1, arr.shape[-1])

    rows_start    = to_rows(start_data)
    rows_off      = to_rows(off_data)
    rows_normal   = to_rows(normal_data)
    rows_high     = to_rows(high_data)
    rows_critical = to_rows(critical_data)
    rows_load     = np.vstack([rows_normal, rows_high, rows_critical])

    X_windows, y_labels = [], []

    # 0 = ENGINE_START (use windows as-is)
    if start_data.ndim == 3 and start_data.shape[1] == 4:
        for window in start_data:
            X_windows.append(window.astype(np.float32))
            y_labels.append(0)
    else:
        raise ValueError(f"{paths['engine_start']} must have shape (N,4,features)")

    # 1 = ENGINE_OFF
    for row in rows_off:
        ctx = []
        for _ in range(3):
            if np.random.rand() < P_MIX_LOAD_IN_OFF:
                # inject Load row instead of Off
                ctx.append(rows_load[np.random.choice(rows_load.shape[0])])
            else:
                ctx.append(rows_off[np.random.choice(rows_off.shape[0])])
        ctx = np.vstack(ctx)
        window = np.vstack([ctx, row.reshape(1, -1)]).astype(np.float32)
        X_windows.append(window)
        y_labels.append(1)

    # 2 = LOAD (Normal/High/Critical) — may mix Start in
    for row in rows_load:
        ctx = []
        for _ in range(3):
            if np.random.rand() < P_MIX_START_IN_LOAD:
                ctx.append(rows_start[np.random.choice(rows_start.shape[0])])
            else:
                ctx.append(rows_load[np.random.choice(rows_load.shape[0])])
        ctx = np.vstack(ctx)
        window = np.vstack([ctx, row.reshape(1, -1)]).astype(np.float32)
        X_windows.append(window)
        y_labels.append(2)

    # Shuffle & save
    X_windows = np.array(X_windows, dtype=np.float32)
    y_labels  = np.array(y_labels,  dtype=np.int32)
    idx = np.arange(len(X_windows))
    np.random.shuffle(idx)
    X_windows, y_labels = X_windows[idx], y_labels[idx]

    split_dir = os.path.join(BASE_PATH, split_name)
    os.makedirs(split_dir, exist_ok=True)
    np.save(os.path.join(split_dir, f"Router_{split_name}_X.npy"), X_windows)
    np.save(os.path.join(split_dir, f"Router_{split_name}_y.npy"), y_labels)

    # Long CSV
    rows = []
    for seq in range(len(X_windows)):
        lab = int(y_labels[seq])
        for t in range(4):
            rows.append({
                "Time": t + 1,
                "Sequence": seq + 1,
                "Temperature": float(X_windows[seq, t, 0]),
                "Pressure":    float(X_windows[seq, t, 1]),
                "RPM":         float(X_windows[seq, t, 2]),
                "Vibration":   float(X_windows[seq, t, 3]),
                "State":       lab
            })
    df = pd.DataFrame(rows, columns=["Time","Sequence",*FEATURES,"State"])
    df.to_csv(os.path.join(split_dir, f"Router_{split_name}.csv"), index=False)

    print(f"Saved {len(X_windows)} windows for {split_name} in {split_dir}")
    return X_windows, y_labels

# Build all splits
for split, paths in DATA_PATHS.items():
    build_router_dataset(split, paths)



--- Building router dataset for split: training ---
Saved 70000 windows for training in ../../../data\training

--- Building router dataset for split: test ---
Saved 13000 windows for test in ../../../data\test

--- Building router dataset for split: final test ---
Saved 13000 windows for final test in ../../../data\final test
